In [2]:
import pandas as pd
import numpy as np

# Look into doing it by maps


In [116]:
df = pd.read_csv('CCRB-Complaint-Data_202007271729/allegations_202007271729.csv')

In [117]:
df = df[['complaint_id', 'month_received', 'year_received', 'command_at_incident', 'rank_abbrev_incident', 'mos_ethnicity', 'mos_gender', 'mos_age_incident', 'complainant_ethnicity', 'complainant_gender', 'fado_type', 'allegation', 'precinct', 'contact_reason', 'outcome_description', 'board_disposition']]
df = df[df['year_received'] <= 2018]
df

,complaint_id,month_received,year_received,command_at_incident,rank_abbrev_incident,mos_ethnicity,mos_gender,mos_age_incident,complainant_ethnicity,complainant_gender,fado_type,allegation,precinct,contact_reason,outcome_description,board_disposition
1,24601,11,2011,PBBS,POM,White,M,24,Black,Male,Discourtesy,Action,67.0,Moving violation,Moving violation summons issued,Substantiated (Charges)
2,24601,11,2011,PBBS,POM,White,M,24,Black,Male,Offensive Language,Race,67.0,Moving violation,Moving violation summons issued,Substantiated (Charges)
3,26146,7,2012,PBBS,POM,White,M,25,Black,Male,Abuse of Authority,Question,67.0,PD suspected C/V of violation/crime - street,No arrest made or summons issued,Substantiated (Charges)
4,40253,8,2018,078 PCT,POF,Hispanic,F,39,NaN,NaN,Force,Physical force,67.0,Report-dispute,Arrest - other violation/crime,Substantiated (Command Discipline A)
5,37256,5,2017,078 PCT,SGT,Black,F,50,White,Male,Abuse of Authority,Refusal to process civilian complaint,78.0,C/V telephoned PCT,No arrest made or summons issued,Substantiated (Command Lvl Instructions)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33353,35671,8,2016,066 PCT,POM,White,M,36,Asian,Male,Discourtesy,Word,66.0,Moving violation,Moving violation summons issued,Unsubstantiated
33354,35671,8,2016,066 PCT,POM,White,M,36,Asian,Male,Abuse of Authority,Interference with recording,66.0,Moving violation,Moving violation summons issued,Unsubstantiated
33355,35671,8,2016,066 PCT,POM,White,M,36,Asian,Male,Abuse of Authority,Search (of person),66.0,Moving violation,Moving violation summons issued,Substantiated (Formalized Training)
33356,35671,8,2016,066 PCT,POM,White,M,36,Asian,Male,Abuse of Authority,Vehicle search,66.0,Moving violation,Moving violation summons issued,Substantiated (Formalized Training)


In [118]:
# combine month and year into a new date column as YYYY-MM 
df['date_received'] = pd.to_datetime(df['year_received'].astype(str) + '-' + df['month_received'].astype(str), format='%Y-%m')
# drop the old month and year columns
df = df.drop(columns=['month_received', 'year_received'])
df['year_month'] = df['date_received'].dt.to_period('M').astype(str)
df['year'] = df['date_received'].dt.year
df

,complaint_id,command_at_incident,rank_abbrev_incident,mos_ethnicity,mos_gender,mos_age_incident,complainant_ethnicity,complainant_gender,fado_type,allegation,precinct,contact_reason,outcome_description,board_disposition,date_received,year_month,year
1,24601,PBBS,POM,White,M,24,Black,Male,Discourtesy,Action,67.0,Moving violation,Moving violation summons issued,Substantiated (Charges),2011-11-01,2011-11,2011
2,24601,PBBS,POM,White,M,24,Black,Male,Offensive Language,Race,67.0,Moving violation,Moving violation summons issued,Substantiated (Charges),2011-11-01,2011-11,2011
3,26146,PBBS,POM,White,M,25,Black,Male,Abuse of Authority,Question,67.0,PD suspected C/V of violation/crime - street,No arrest made or summons issued,Substantiated (Charges),2012-07-01,2012-07,2012
4,40253,078 PCT,POF,Hispanic,F,39,NaN,NaN,Force,Physical force,67.0,Report-dispute,Arrest - other violation/crime,Substantiated (Command Discipline A),2018-08-01,2018-08,2018
5,37256,078 PCT,SGT,Black,F,50,White,Male,Abuse of Authority,Refusal to process civilian complaint,78.0,C/V telephoned PCT,No arrest made or summons issued,Substantiated (Command Lvl Instructions),2017-05-01,2017-05,2017
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33353,35671,066 PCT,POM,White,M,36,Asian,Male,Discourtesy,Word,66.0,Moving violation,Moving violation summons issued,Unsubstantiated,2016-08-01,2016-08,2016
33354,35671,066 PCT,POM,White,M,36,Asian,Male,Abuse of Authority,Interference with recording,66.0,Moving violation,Moving violation summons issued,Unsubstantiated,2016-08-01,2016-08,2016
33355,35671,066 PCT,POM,White,M,36,Asian,Male,Abuse of Authority,Search (of person),66.0,Moving violation,Moving violation summons issued,Substantiated (Formalized Training),2016-08-01,2016-08,2016
33356,35671,066 PCT,POM,White,M,36,Asian,Male,Abuse of Authority,Vehicle search,66.0,Moving violation,Moving violation summons issued,Substantiated (Formalized Training),2016-08-01,2016-08,2016


In [119]:
import pandas as pd
import plotly.express as px

# Filter to relevant races
target_races = ['Black', 'White', 'Asian']
filtered_df = df[df['complainant_ethnicity'].isin(target_races)]

# Get top 10 allegations by overall frequency
top_allegations = filtered_df['allegation'].value_counts().nlargest(5).index
filtered_df = filtered_df[filtered_df['allegation'].isin(top_allegations)]

# Total complaints per race (for normalization)
total_by_race = filtered_df.groupby('complainant_ethnicity').size().to_dict()

# Group and calculate proportion
grouped = (
    filtered_df.groupby(['allegation', 'complainant_ethnicity'])
    .size()
    .reset_index(name='count')
)

# Add proportion column
grouped['proportion'] = grouped.apply(
    lambda row: row['count'] / total_by_race[row['complainant_ethnicity']],
    axis=1
)

# Plot
fig = px.bar(
    grouped,
    x='allegation',
    y='proportion',
    color='complainant_ethnicity',
    barmode='group',
    title='Proportion of Each Race’s Complaints by Allegation Type',
    labels={
        'allegation': 'Allegation Type',
        'proportion': 'Proportion of Race’s Complaints',
        'complainant_ethnicity': 'Race'
    }
)

fig.update_layout(xaxis_tickangle=-45, yaxis_tickformat='.0%')
fig.show()


In [120]:
import pandas as pd
import plotly.express as px

# Filter to relevant races
target_races = ['Black', 'White', 'Asian']
filtered_df = df[df['complainant_ethnicity'].isin(target_races)]

# Get top 5 allegations by overall frequency
top_allegations = filtered_df['allegation'].value_counts().nlargest(5).index
filtered_df = filtered_df[filtered_df['allegation'].isin(top_allegations)]

# Total complaints per race
total_by_race = filtered_df.groupby('complainant_ethnicity').size().to_dict()

# Count by allegation + race
grouped = (
    filtered_df.groupby(['allegation', 'complainant_ethnicity'])
    .size()
    .reset_index(name='count')
)

# Add proportion
grouped['proportion'] = grouped.apply(
    lambda row: row['count'] / total_by_race[row['complainant_ethnicity']],
    axis=1
)

# Ensure ordering is consistent
race_order = ['Black', 'White', 'Asian']
grouped['complainant_ethnicity'] = pd.Categorical(grouped['complainant_ethnicity'], categories=race_order, ordered=True)

# Plot with custom colors
fig = px.bar(
    grouped,
    x='allegation',
    y='proportion',
    color='complainant_ethnicity',
    category_orders={'complainant_ethnicity': race_order},
    color_discrete_map={
        'Black': '#ff5b55',
        'White': '#ff9c2d',
        'Asian': '#93c7f7'
    },
    barmode='group',
    title='Proportion of Each Race’s Complaints by Allegation Type',
    labels={
        'allegation': 'Allegation Type',
        'proportion': 'Proportion of Race’s Complaints',
        'complainant_ethnicity': 'Race'
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    yaxis_tickformat='.0%',
    plot_bgcolor='white',
    yaxis=dict(
        gridcolor='lightgray',
        range=[0, 0.4]  # ✅ sets y-axis to go from 0 to 40%
    ),
    bargroupgap=0.05
)
fig.show()


In [121]:
import pandas as pd
import plotly.express as px

# Filter by your selected races
races = ['Black', 'White', 'Asian']

# Drop missing race/allegation values and filter by race
filtered = df[
    df['complainant_ethnicity'].isin(races) &
    df['allegation'].notna()
].copy()

# Focus on top 5 most common allegations
top_5 = (
    filtered['allegation']
    .value_counts()
    .head(5)
    .index
)
filtered = filtered[filtered['allegation'].isin(top_5)]

# Count complaints by allegation + race
counts = (
    filtered.groupby(['allegation', 'complainant_ethnicity'])
    .size()
    .reset_index(name='count')
)

# Convert race column to categorical with desired order
counts['complainant_ethnicity'] = pd.Categorical(
    counts['complainant_ethnicity'],
    categories=['Black', 'White', 'Asian'],
    ordered=True
)

# Get total complaints per allegation to compute proportion
totals = counts.groupby('allegation')['count'].transform('sum')
counts['proportion'] = counts['count'] / totals

# Define custom color mapping
color_map = {
    'Black': '#fa3b1c',
    'White': '#867271',
    'Asian': '#c4bbbb'
}

# Plot
fig = px.bar(
    counts.sort_values('complainant_ethnicity'),  # ensures bar order within each group
    x='allegation',
    y='proportion',
    color='complainant_ethnicity',
    color_discrete_map=color_map,
    barmode='group',
    title='Proportion of Complaint Races per Allegation Type',
    labels={
        'allegation': 'Allegation Type',
        'proportion': 'Proportion of Complaints',
        'complainant_ethnicity': 'Race'
    }
)

# Clean up appearance
fig.update_layout(
    xaxis_tickangle=-30,
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor='lightgray')
)

fig.show()


In [124]:
import pandas as pd
import plotly.express as px

# Filter to only Black, White, Asian complainants
target_races = ['Black', 'White', 'Asian']
filtered_df = df[df['complainant_ethnicity'].isin(target_races)].copy()

# Get top 5 most common allegations
top_allegations = filtered_df['allegation'].value_counts().nlargest(15).index
filtered_df = filtered_df[filtered_df['allegation'].isin(top_allegations)]

# Flag whether the complaint was substantiated
filtered_df['is_substantiated'] = filtered_df['board_disposition'].str.contains('Substantiated', case=False, na=False)

# Group and calculate substantiation rate
grouped = (
    filtered_df.groupby(['allegation', 'complainant_ethnicity'])['is_substantiated']
    .mean()
    .reset_index(name='substantiated_rate')
)

# Set race order
race_order = ['Black', 'White', 'Asian']
grouped['complainant_ethnicity'] = pd.Categorical(grouped['complainant_ethnicity'], categories=race_order, ordered=True)

# Create bar chart
fig = px.bar(
    grouped,
    x='allegation',
    y='substantiated_rate',
    color='complainant_ethnicity',
    barmode='group',
    color_discrete_map={
        'Black': '#fa3b1c',
        'White': '#867271',
        'Asian': '#c4bbbb'
    },
    title='Substantiation Rate by Allegation Type and Race',
    labels={
        'allegation': 'Allegation Type',
        'substantiated_rate': 'Proportion Substantiated',
        'complainant_ethnicity': 'Race'
    }
)

# Style updates
fig.update_layout(
    yaxis_tickformat='.0%',
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgray'),
    bargroupgap=0.15
)

fig.show()


In [ ]:
# Frisk, Physical force, search, threat of force, Other

In [129]:
import pandas as pd
import plotly.express as px

# Filter to selected races
target_races = ['Black', 'White', 'Asian']
filtered_df = df[df['complainant_ethnicity'].isin(target_races)].copy()

# Keep only these 5 allegations
target_allegations = ['Frisk', 'Physical force', 'Search (of person)', 'Threat of force (verbal or physical)', 'Other']
filtered_df = filtered_df[filtered_df['allegation'].isin(target_allegations)]

# Flag substantiated complaints
filtered_df['is_substantiated'] = filtered_df['board_disposition'].str.contains('Substantiated', case=False, na=False)

# Group and calculate proportion substantiated
grouped = (
    filtered_df.groupby(['allegation', 'complainant_ethnicity'])['is_substantiated']
    .mean()
    .reset_index(name='substantiated_rate')
)

# Set desired order for race and allegation
race_order = ['Black', 'White', 'Asian']
allegation_order = ['Physical force', 'Search (of person)', 'Threat of force (verbal or physical)', 'Frisk', 'Other']

grouped['complainant_ethnicity'] = pd.Categorical(grouped['complainant_ethnicity'], categories=race_order, ordered=True)
grouped['allegation'] = pd.Categorical(grouped['allegation'], categories=allegation_order, ordered=True)

# Plot
fig = px.bar(
    grouped,
    x='allegation',
    y='substantiated_rate',
    color='complainant_ethnicity',
    barmode='group',
    color_discrete_map={
        'Black': '#ff5b55',
        'White': '#ff9c2d',
        'Asian': '#93c7f7'
    },
    title='Substantiation Rate by Allegation and Race (Selected Categories)',
    labels={
        'allegation': 'Allegation Type',
        'substantiated_rate': 'Proportion Substantiated',
        'complainant_ethnicity': 'Race'
    }
)

fig.update_layout(
    yaxis_tickformat='.0%',
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgray'),
    bargroupgap=0.15
)

fig.show()


In [130]:
import pandas as pd
import plotly.express as px

# Filter to female complainants of specific races
target_races = ['Black', 'White', 'Asian']
filtered_df = df[
    (df['complainant_ethnicity'].isin(target_races)) &
    (df['complainant_gender'].str.lower() == 'female')
].copy()

# Count complaints per race
grouped = (
    filtered_df.groupby('complainant_ethnicity')
    .size()
    .reset_index(name='count')
)

# Calculate proportions
grouped['proportion'] = grouped['count'] / grouped['count'].sum()

# Set race order
race_order = ['Black', 'White', 'Asian']
grouped['complainant_ethnicity'] = pd.Categorical(grouped['complainant_ethnicity'], categories=race_order, ordered=True)

# Plot
fig = px.bar(
    grouped.sort_values('complainant_ethnicity'),
    x='complainant_ethnicity',
    y='proportion',
    color='complainant_ethnicity',
    color_discrete_map={
        'Black': '#fa3b1c',
        'White': '#867271',
        'Asian': '#c4bbbb'
    },
    title='Proportion of NYPD Complaints Filed by Women (by Race)',
    labels={
        'complainant_ethnicity': 'Race',
        'proportion': 'Proportion of Complaints'
    }
)

fig.update_layout(
    yaxis_tickformat='.0%',
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgray', range=[0, 1]),
    showlegend=False
)

fig.show()


In [131]:
df

,complaint_id,command_at_incident,rank_abbrev_incident,mos_ethnicity,mos_gender,mos_age_incident,complainant_ethnicity,complainant_gender,fado_type,allegation,precinct,contact_reason,outcome_description,board_disposition,date_received,year_month,year
1,24601,PBBS,POM,White,M,24,Black,Male,Discourtesy,Action,67.0,Moving violation,Moving violation summons issued,Substantiated (Charges),2011-11-01,2011-11,2011
2,24601,PBBS,POM,White,M,24,Black,Male,Offensive Language,Race,67.0,Moving violation,Moving violation summons issued,Substantiated (Charges),2011-11-01,2011-11,2011
3,26146,PBBS,POM,White,M,25,Black,Male,Abuse of Authority,Question,67.0,PD suspected C/V of violation/crime - street,No arrest made or summons issued,Substantiated (Charges),2012-07-01,2012-07,2012
4,40253,078 PCT,POF,Hispanic,F,39,NaN,NaN,Force,Physical force,67.0,Report-dispute,Arrest - other violation/crime,Substantiated (Command Discipline A),2018-08-01,2018-08,2018
5,37256,078 PCT,SGT,Black,F,50,White,Male,Abuse of Authority,Refusal to process civilian complaint,78.0,C/V telephoned PCT,No arrest made or summons issued,Substantiated (Command Lvl Instructions),2017-05-01,2017-05,2017
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33353,35671,066 PCT,POM,White,M,36,Asian,Male,Discourtesy,Word,66.0,Moving violation,Moving violation summons issued,Unsubstantiated,2016-08-01,2016-08,2016
33354,35671,066 PCT,POM,White,M,36,Asian,Male,Abuse of Authority,Interference with recording,66.0,Moving violation,Moving violation summons issued,Unsubstantiated,2016-08-01,2016-08,2016
33355,35671,066 PCT,POM,White,M,36,Asian,Male,Abuse of Authority,Search (of person),66.0,Moving violation,Moving violation summons issued,Substantiated (Formalized Training),2016-08-01,2016-08,2016
33356,35671,066 PCT,POM,White,M,36,Asian,Male,Abuse of Authority,Vehicle search,66.0,Moving violation,Moving violation summons issued,Substantiated (Formalized Training),2016-08-01,2016-08,2016


In [143]:
import pandas as pd

# Filter to relevant races
target_races = ['Black', 'White', 'Hispanic']
filtered_df = df[df['complainant_ethnicity'].isin(target_races)].copy()
filter_df = filtered_df[filtered_df['complainant_gender'] == 'Female']

# Group by fado_type and race, then pivot into columns
race_counts = (
    filtered_df.groupby(['allegation', 'complainant_ethnicity'])
    .size()
    .unstack(fill_value=0)  # fills missing race values with 0
    .reset_index()
)

# Optional: reorder columns
race_counts = race_counts[['allegation'] + target_races]

# Display the table
print(race_counts)


complainant_ethnicity            allegation  Black  White  Hispanic
0                                    Action    152     31        66
1                                    Animal      1      0         2
2                      Body Cavity Searches      1      1         0
3                                 Chokehold    146     15        59
4                                     Curse      1      0         1
..                                      ...    ...    ...       ...
74                     Threat to notify ACS      2      1         2
75                                  Vehicle     20      3         7
76                           Vehicle search    806    133       275
77                             Vehicle stop    648    101       204
78                                     Word   2014    416       894

[79 rows x 4 columns]


In [145]:
race_counts.loc[20:40]

complainant_ethnicity,allegation,Black,White,Hispanic
20,Gun as club,17,3,5
21,Gun fired,30,0,3
22,Gun pointed,1,0,0
23,Gun pointed/gun drawn,2,0,1
24,Handcuffs too tight,45,7,15
25,Hit against inanimate object,91,16,45
26,Improper dissemination of medical info,1,0,1
27,Interference with recording,45,9,20
28,Nightstick as club (incl asp & baton),152,12,66
29,Nonlethal restraining device,48,6,16


In [ ]:
# row 8, ethnicity

In [159]:
df['complainant_ethnicity'].value_counts()

complainant_ethnicity
Black              16380
Hispanic            6085
White               2605
Unknown              983
Other Race           600
Asian                486
Refused              236
American Indian       56
Name: count, dtype: int64

In [160]:
# Choose the races you're interested in
target_races = ['Black', 'White', 'Hispanic']

# Define how many you want from each
n_per_race = 2000  # adjust this based on the lowest group's size (White = 2605 in your case)

# Sample from each group equally
balanced_samples = (
    df[df['complainant_ethnicity'].isin(target_races)]
    .groupby('complainant_ethnicity')
    .apply(lambda x: x.sample(n=n_per_race, replace=False, random_state=42))
    .reset_index(drop=True)
)


/var/folders/nw/0j298g817b92m2qlhw1_fwnr0000gn/T/ipykernel_54091/2000702973.py:11: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [164]:
import pandas as pd
import plotly.express as px

# Step 1: Define target races and sample size
target_races = ['Black', 'White', 'Hispanic']
n_per_race = 2000

# Step 2: Sample with replacement to get a balanced dataset
balanced_samples = (
    df[df['complainant_ethnicity'].isin(target_races)]
    .groupby('complainant_ethnicity')
    .apply(lambda x: x.sample(n=n_per_race, replace=True, random_state=42))
    .reset_index(drop=True)
)

# Step 3: Get top 5 allegations in this sample
top_5_allegations = (
    balanced_samples['allegation']
    .value_counts()
    .head(22)
    .index
)
filtered = balanced_samples[balanced_samples['allegation'].isin(top_5_allegations)]

# Step 4: Count complaints by allegation and race
counts = (
    filtered.groupby(['allegation', 'complainant_ethnicity'])
    .size()
    .reset_index(name='count')
)

# Step 5: Compute proportions per allegation group
totals = counts.groupby('allegation')['count'].transform('sum')
counts['proportion'] = counts['count'] / totals

# Step 6: Plot
fig = px.bar(
    counts,
    x='allegation',
    y='proportion',
    color='complainant_ethnicity',
    barmode='group',
    color_discrete_map={
        'Black': '#ff5b55',
        'White': '#ff9c2d',
        'Hispanic': '#93c7f7'
    },
    category_orders={'complainant_ethnicity': ['Black', 'White', 'Hispanicn']},
    title='Proportion of Races per Allegation Type (Sampled 1000 Each)',
    labels={
        'allegation': 'Allegation Type',
        'proportion': 'Proportion of Allegation',
        'complainant_ethnicity': 'Race'
    }
)

fig.update_layout(
    yaxis_tickformat='.0%',
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgray'),
    bargroupgap=0.15
)

fig.show()


/var/folders/nw/0j298g817b92m2qlhw1_fwnr0000gn/T/ipykernel_54091/813466432.py:12: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [ ]:
Search, vehicle search, vehicle stop, word, physical force, frisk

In [172]:
import pandas as pd
import plotly.express as px

# Step 1: Define target races and sample size
target_races = ['Black', 'White', 'Hispanic']
n_per_race = 2000

# Step 2: Sample with replacement to get a balanced dataset
balanced_samples = (
    df[df['complainant_ethnicity'].isin(target_races)]
    .groupby('complainant_ethnicity')
    .apply(lambda x: x.sample(n=n_per_race, replace=True, random_state=42))
    .reset_index(drop=True)
)

# Step 3: Filter to only specific allegations
selected_allegations = [
    'Frisk',
    'Physical force',
    'Search (of person)',
    'Vehicle stop',
    'Word'
]
filtered = balanced_samples[balanced_samples['allegation'].isin(selected_allegations)]

# Step 4: Count complaints by allegation and race
counts = (
    filtered.groupby(['allegation', 'complainant_ethnicity'])
    .size()
    .reset_index(name='count')
)

# Step 5: Compute proportions per allegation group
totals = counts.groupby('allegation')['count'].transform('sum')
counts['proportion'] = counts['count'] / totals

# Step 6: Plot
fig = px.bar(
    counts,
    x='allegation',
    y='proportion',
    color='complainant_ethnicity',
    barmode='group',
    color_discrete_map={
        'Black': '#ff5b55',
        'White': '#ff9c2d',
        'Hispanic': '#93c7f7'
    },
    category_orders={
        'complainant_ethnicity': ['Black', 'White', 'Hispanic'],
        'allegation': selected_allegations  # optional: keep x-axis in defined order
    },
    title='Proportion of Races per Allegation Type (Sampled 2000 Each)',
    labels={
        'allegation': 'Allegation Type',
        'proportion': 'Proportion of Allegation',
        'complainant_ethnicity': 'Race'
    }
)

fig.update_layout(
    yaxis_tickformat='.0%',
    plot_bgcolor='white',
    yaxis=dict(gridcolor='lightgray', range = [0, .7]),
    bargroupgap=0.15
)

fig.show()


/var/folders/nw/0j298g817b92m2qlhw1_fwnr0000gn/T/ipykernel_54091/3345674487.py:12: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

